In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load and preprocess data
data = pd.read_parquet("sales_features.parquet").reset_index()
data = data.sort_values(["store", "date"])
data["time_idx"] = data.groupby("store").cumcount() + 1

# Filter stores with sufficient data
store_time_counts = data.groupby("store")["time_idx"].count()
valid_stores = store_time_counts[store_time_counts >= 30 + 7].index
data = data[data["store"].isin(valid_stores)]

# Feature configuration
categorical_features = ["store", "month", "is_holiday"]
numeric_features = [
    "sale_dollars",  # Will be used as a predictor (scaled)
    "day_of_month", 
    "sale_dollars_rolling_mean_7D",
    "days_to_nearest_holiday"
]

# Encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Preserve original target values before scaling
data["original_sale_dollars"] = data["sale_dollars"]

# Normalize predictors (including historical sale_dollars)
scaler = StandardScaler()
data[numeric_features] = scaler.fit_transform(data[numeric_features])

# Create sequences with scaled predictors and original targets
def create_sequences(data, window_size, forecast_horizon):
    sequences = []
    targets = []
    stores = data["store"].unique()
    
    for store in stores:
        store_data = data[data["store"] == store].sort_values("time_idx")
        predictors = store_data[numeric_features + categorical_features].values
        target_vals = store_data["original_sale_dollars"].values  # Unscaled targets
        
        for i in range(len(predictors) - window_size - forecast_horizon):
            sequences.append(predictors[i:i+window_size])
            targets.append(target_vals[i+window_size:i+window_size+forecast_horizon])
            
    return np.array(sequences), np.array(targets)

X, y = create_sequences(data, window_size=30, forecast_horizon=7)

# Split data
split_idx = int(0.8 * len(X))
train_X, test_X = X[:split_idx], X[split_idx:]
train_y, test_y = y[:split_idx], y[split_idx:]

# Model architecture
def build_temporal_model(input_shape, forecast_horizon):
    inputs = layers.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=True)(inputs)
    x = layers.LSTM(32)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(forecast_horizon)(x)
    
    model = models.Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

model = build_temporal_model(
    input_shape=(30, len(numeric_features) + len(categorical_features)),
    forecast_horizon=7
)

# Training with early stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    train_X, train_y,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop]
)

# Evaluation
test_results = model.evaluate(test_X, test_y)
print(f"Test RMSE: {test_results[1]:.2f}")

# Generate predictions
test_preds = model.predict(test_X)

Test RMSE: 812.36
